# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [31]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [32]:
import pandas as pd

In [33]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
232,I love this magazine and read every word print...,good
922,I find the fashions terrible and will end my f...,bad
730,There are way too many cigarettes and alcohol ...,bad
54,"Boyfriend loves these, will renew",good
854,Not able to download on Paperwhite!!! What els...,bad


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [34]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [35]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [36]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [37]:
from sklearn import metrics

In [38]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [39]:
evaluate(sgd_clf)

Accuracy: 77.1%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [40]:
from sklearn.neighbors import KNeighborsClassifier

kn_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KNeighborsClassifier()),
])

_ = kn_clf.fit(train['review'], train['label'])

def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

evaluate(kn_clf)

Accuracy: 78.6%


In [41]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', DecisionTreeClassifier()),
])

_ = dt_clf.fit(train['review'], train['label'])

def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

evaluate(dt_clf)

Accuracy: 77.7%


In [42]:
from sklearn.ensemble import AdaBoostClassifier

ab_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', AdaBoostClassifier()),
])

_ = ab_clf.fit(train['review'], train['label'])

def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

evaluate(ab_clf)

Accuracy: 71.9%


In [43]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', RandomForestClassifier()),
])

_ = rf_clf.fit(train['review'], train['label'])

def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

evaluate(rf_clf)

Accuracy: 84.1%


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [44]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


Zooming in on one particular data point:

In [45]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [46]:
train.sample(n=10)

,review,label
2775,Sent this as a gift and the friend loved it!</...,bad
1683,This magazine is not practical. I should have ...,bad
3930,I love this magazine. Always tons of great rec...,bad
1086,Iv read this book from cover to cover absolute...,good
3421,"Wonderful mag, A plus always. Thanks.",good
6472,Never received full order. Received a couple m...,bad
412,It was a gift for my son in law who loves anyt...,good
5152,National Geographic is still the best way to s...,good
3624,I did not want to renew this item and did not ...,bad
4873,"<li class=""api apilevel-"">I thought this was t...",good


## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [47]:
def is_bad_data(review: str) -> bool:
    # Heuristic: Check for the presence of both '<' and '>' in the review string
    return '<' in review and '>' in review

## Creating the cleaned training set

In [48]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [49]:
from sklearn import clone

In [50]:
sgd_clf_clean = clone(sgd_clf)

In [51]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [52]:
evaluate(sgd_clf_clean)

Accuracy: 96.9%


In [53]:
kn_clf_clean = clone(kn_clf)
_ = kn_clf_clean.fit(train_clean['review'], train_clean['label'])
evaluate(kn_clf_clean)

Accuracy: 91.1%


In [54]:
dt_clf_clean = clone(dt_clf)
_ = dt_clf_clean.fit(train_clean['review'], train_clean['label'])
evaluate(dt_clf_clean)

Accuracy: 91.8%


In [55]:
ab_clf_clean = clone(ab_clf)
_ = ab_clf_clean.fit(train_clean['review'], train_clean['label'])
evaluate(ab_clf_clean)

Accuracy: 88.3%


In [56]:
rf_clf_clean = clone(rf_clf)
_ = rf_clf_clean.fit(train_clean['review'], train_clean['label'])
evaluate(rf_clf_clean)

Accuracy: 95.0%
